In [2]:
# peft: to load and merge the adapter. langchain-openai + python-dotenv: only so the
# repo's own prompt builder (agents.critic) can be imported for the sanity check.
!pip install -q -U peft
!pip install -q langchain-openai python-dotenv
!git clone -q https://github.com/akshatyuvan/autonomous-preprocessing-agent.git /tmp/repo
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip uninstall -y torchao

fatal: destination path '/tmp/repo' already exists and is not an empty directory.
fatal: destination path '/tmp/llama.cpp' already exists and is not an empty directory.


In [3]:
import json, os, re, sys
from pathlib import Path

os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
# The EXACT base model version the adapter was trained on (evaluation/kaggle/base_model_revision.json).
REVISION = "0cb88a4f764b7a12671c53f0838cd831a0843b95"
MERGED = Path("/tmp/merged")          # /tmp, not /kaggle/working: 6.4 GB we don't want as output
OUT = Path("/kaggle/working")

# Find the adapter in whatever folder Kaggle mounted the critic-qlora output under.
ADAPTER = next(Path("/kaggle/input").rglob("adapter_config.json")).parent
print("adapter:", ADAPTER, sorted(p.name for p in ADAPTER.iterdir()))

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

adapter: /kaggle/input/notebooks/akshatyuvan/critic-qlora/critic_lora_adapter ['README.md', 'adapter_config.json', 'adapter_model.safetensors']


In [4]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
# Full precision (fp16), NOT 4-bit: merging means adding the LoRA product B@A into each
# target weight matrix, which needs real numbers to add to. 4-bit weights can't hold the sum.
# Known mismatch: the adapter was TRAINED on the 4-bit NF4 version of these weights, so
# merging into fp16 shifts things slightly. Cell 4 measures whether that matters.
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION,
                                            dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(base, str(ADAPTER))
# merge_and_unload: W_new = W + (alpha/r) * B @ A for every adapted layer, then the LoRA
# wrappers are removed. Result: an ordinary Llama model that any tool can load, with
# zero extra inference cost (no adapter computed at run time).
model = model.merge_and_unload()
model.eval()
print("merged;", torch.cuda.memory_allocated() / 1e9, "GB on GPU")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

merged; 6.43454464 GB on GPU


In [6]:
# Same prompt format and parser as the Kaggle evaluation, on the first 16 test rows.
# Not a new result for the table -- just proof the merge didn't break the model.
from agents.critic import DecisionJudgment, build_critic_messages

FORMAT = ('\n\nRespond with ONLY a JSON object and nothing else: '
          '{"reason": "<one sentence citing the rule number and the numbers>", '
          '"verdict": "accept" or "reject"}')

def chat(evidence):
    (_, system), (_, human) = build_critic_messages(evidence)
    return [{"role": "system", "content": system}, {"role": "user", "content": human + FORMAT}]

def parse(text):
    m = re.search(r'"verdict"\s*:\s*"([^"]*)"', text)
    try:
        return DecisionJudgment(verdict=m.group(1), reason="").verdict if m else None
    except Exception:
        return None

test = [json.loads(l) for l in open("evaluation/data/critic_test.jsonl") if l.strip()][:16]
tok.pad_token = tok.eos_token  # Llama has no pad token; padded positions are masked out anyway
prompts = [tok.apply_chat_template(chat(e["evidence"]), tokenize=False, add_generation_prompt=True)
           for e in test]
correct = 0
with torch.no_grad():
    for i in range(0, len(prompts), 8):
        enc = tok(prompts[i:i + 8], return_tensors="pt", padding=True,
                  add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=80, do_sample=False,
                             pad_token_id=tok.eos_token_id)
        for e, text in zip(test[i:i + 8], tok.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                                           skip_special_tokens=True)):
            correct += parse(text) == e["label"]
print(f"merged model: {correct}/16 correct on the first 16 test rows")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


merged model: 16/16 correct on the first 16 test rows


In [7]:
model.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)  # includes the chat template, which Ollama needs to format prompts
del model, base
torch.cuda.empty_cache()
print(sorted(p.name for p in MERGED.iterdir()))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


In [8]:
# llama.cpp's converter turns a Hugging Face model folder into GGUF, the single-file
# format Ollama runs. Its requirements may downgrade Kaggle's transformers/torch; that's
# fine now, because the merge is already saved to disk. Expect pip conflict warnings.
!pip install -q -r /tmp/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile /tmp/critic-ft-f16.gguf --outtype f16
# Build only the quantize tool (a few minutes). CUDA off: compression runs on the CPU.
!cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /tmp/llama.cpp/build --target llama-quantize -j 4 > /dev/null
# Q4_K_M = the same 4-bit scheme Ollama's own llama3.2:3b uses: ~6.4 GB -> ~2 GB.
!/tmp/llama.cpp/build/bin/llama-quantize /tmp/critic-ft-f16.gguf /kaggle/working/critic-ft-q4_k_m.gguf Q4_K_M
!ls -lh /kaggle/working/critic-ft-q4_k_m.gguf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 81.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you ha

In [9]:
import shutil
from huggingface_hub import snapshot_download

# Merging only changes weights, never the tokenizer, so Meta's ORIGINAL tokenizer files
# (same pinned revision) are exactly right. They're already in the local cache.
TOKENIZER_FILES = ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]
src = Path(snapshot_download(MODEL_ID, revision=REVISION, allow_patterns=TOKENIZER_FILES))
for name in TOKENIZER_FILES:
    shutil.copy(src / name, MERGED / name)
print(json.load(open(MERGED / "tokenizer_config.json"))["tokenizer_class"])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

PreTrainedTokenizerFast


In [10]:
!python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile /tmp/critic-ft-f16.gguf --outtype f16
!/tmp/llama.cpp/build/bin/llama-quantize /tmp/critic-ft-f16.gguf /kaggle/working/critic-ft-q4_k_m.gguf Q4_K_M
!ls -lh /kaggle/working/critic-ft-q4_k_m.gguf

INFO:hf-to-gguf:Loading model: merged
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,    torc

In [11]:
# Upload the GGUF to a PRIVATE repo on your Hugging Face account, then download it
# from huggingface.co (a CDN built for multi-GB files) instead of Kaggle's file proxy.
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

api = HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN"))
# Llama 3.2's license asks derived models to start their name with "Llama"; private=True
# means only you can see or download it.
repo_id = api.create_repo("Llama-3.2-3B-critic-ft-GGUF", private=True, exist_ok=True).repo_id
api.upload_file(path_or_fileobj="/kaggle/working/critic-ft-q4_k_m.gguf",
                path_in_repo="critic-ft-q4_k_m.gguf", repo_id=repo_id)
print(f"https://huggingface.co/{repo_id}/tree/main")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

https://huggingface.co/akshatyuvan/Llama-3.2-3B-critic-ft-GGUF/tree/main
